# Project 8 — Tiny Object Detector (boxes, IoU, NMS + tiny head)
```
image(64x64x3) -> tiny CNN -> box head (4: cx,cy,w,h sigmoid) + class head (3 logits)
                        +
utils from scratch: pairwise IoU, NMS, box encode/decode
```
**Goal:** demystify detection: boxes are just regressed numbers, IoU/NMS are plain geometry.
**Objectives:** (1) implement IoU + NMS, unit-test vs hand values; (2) train a 1-object
detector on synthetic shapes; (3) report mean-IoU + precision@0.5; (4) visualize GT vs pred.

In [ ]:
import os # need os to create results/
import random # need random to seed the stdlib RNG
import time # need time to time training and NMS
import csv # need csv to save metrics tables
import numpy as np # need numpy for boxes, IoU, NMS, and synthetic data
import torch # need torch for the detector model and training
import torch.nn as nn # need nn for conv/linear modules
import torch.nn.functional as F # need F for relu/pool/losses
import matplotlib # need matplotlib for plots
matplotlib.use('Agg') # headless backend so nbconvert execute works
import matplotlib.pyplot as plt # need pyplot for curves and box overlays
import matplotlib.patches as patches # need patches to draw GT/pred rectangles
SEED = 8 # fixed seed for reproducibility
random.seed(SEED) # seed stdlib
np.random.seed(SEED) # seed numpy
torch.manual_seed(SEED) # seed torch
torch.set_num_threads(4) # cap threads for stable timing
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # GPU if present else CPU
print('device:', DEVICE) # log device
os.makedirs('./results', exist_ok=True) # create results/ for CSVs and plots

In [ ]:
IMG = 64 # synthetic image size 64x64
NCLASS = 3 # three shape classes: 0=square(red) 1=tall-rect(green) 2=wide-rect(blue)
NTRAIN, NVAL = 1500, 300 # dataset sizes: tiny, trains in seconds on CPU
COLORS = np.array([[1.0, 0.15, 0.15], [0.15, 0.85, 0.25], [0.25, 0.45, 1.0]], dtype=np.float32) # class colors
def render_one(gen): # render one image: gray bg + one colored rectangle + its box
    img = np.full((IMG, IMG, 3), 0.5, dtype=np.float32) # mid-gray background
    c = int(gen.integers(0, NCLASS)) # random class 0..2
    bw = float(gen.uniform(10, 26)) # random box width in pixels
    bh = float(gen.uniform(10, 26)) if c == 0 else float(gen.uniform(10, 26)) * (0.6 if c == 1 else 1.4) # class 1 tall-ish, 2 wide-ish
    bh = float(np.clip(bh, 8, 30)) # keep height inside sane bounds
    cx = float(gen.uniform(bw/2+1, IMG-bw/2-1)) # random center x clear of borders
    cy = float(gen.uniform(bh/2+1, IMG-bh/2-1)) # random center y clear of borders
    x0, y0 = int(cx-bw/2), int(cy-bh/2) # top-left corner in pixels
    x1, y1 = int(cx+bw/2), int(cy+bh/2) # bottom-right corner in pixels
    img[y0:y1, x0:x1] = COLORS[c] # paint the rectangle
    img += gen.normal(0, 0.02, img.shape).astype(np.float32) # add slight sensor noise
    box = np.array([x0/IMG, y0/IMG, x1/IMG, y1/IMG], dtype=np.float32) # normalized xyxy GT box
    return np.clip(img, 0, 1), box, c # clipped image plus targets
def render_set(n, seed): # render a full split deterministically
    gen = np.random.default_rng(seed) # dedicated RNG per split
    X = np.zeros((n, IMG, IMG, 3), dtype=np.float32) # image buffer
    B = np.zeros((n, 4), dtype=np.float32) # box buffer (xyxy normalized)
    Y = np.zeros(n, dtype=np.int64) # label buffer
    for i in range(n): # render each sample
        X[i], B[i], Y[i] = render_one(gen) # unpack image, box, class
    return X, B, Y # return the split
Xtr, Btr, ytr = render_set(NTRAIN, SEED) # train split
Xva, Bva, yva = render_set(NVAL, SEED+1) # val split with a different seed
print('train', Xtr.shape, Btr.shape, 'val', Xva.shape) # log split shapes
Xtr = Xtr.transpose(0, 3, 1, 2).copy() # HWC -> CHW for torch
Xva = Xva.transpose(0, 3, 1, 2).copy() # HWC -> CHW for torch
BATCH = 64 # minibatch size
tr_ld = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(Btr), torch.from_numpy(ytr)), batch_size=BATCH, shuffle=True, num_workers=0) # shuffled train loader
va_ld = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(torch.from_numpy(Xva), torch.from_numpy(Bva), torch.from_numpy(yva)), batch_size=256, shuffle=False, num_workers=0) # plain val loader

In [ ]:
def iou_pair(a, b): # IoU of two xyxy boxes, pure python floats
    ix0, iy0 = max(a[0], b[0]), max(a[1], b[1]) # intersection top-left
    ix1, iy1 = min(a[2], b[2]), min(a[3], b[3]) # intersection bottom-right
    iw, ih = max(0.0, ix1-ix0), max(0.0, iy1-iy0) # clamp negative overlap to zero
    inter = iw*ih # intersection area
    area_a = max(0.0, a[2]-a[0])*max(0.0, a[3]-a[1]) # area of box a
    area_b = max(0.0, b[2]-b[0])*max(0.0, b[3]-b[1]) # area of box b
    union = area_a + area_b - inter # union = sum minus overlap
    return inter/union if union > 0 else 0.0 # guard the degenerate zero-area case
print('iou self-test:', round(iou_pair([0,0,1,1],[0.5,0.5,1.5,1.5]), 6), '(expect 0.142857)') # 0.25/1.75 = 1/7
assert abs(iou_pair([0,0,1,1],[0.5,0.5,1.5,1.5]) - 1/7) < 1e-9 # fail loudly on wrong geometry
assert iou_pair([0,0,1,1],[0,0,1,1]) == 1.0 # identical boxes must give 1
assert iou_pair([0,0,1,1],[2,2,3,3]) == 0.0 # disjoint boxes must give 0
def iou_matrix(boxes1, boxes2): # vectorized NxM IoU via broadcasting
    b1 = np.asarray(boxes1, dtype=np.float64) # (N,4) array
    b2 = np.asarray(boxes2, dtype=np.float64) # (M,4) array
    ix0 = np.maximum(b1[:, None, 0], b2[None, :, 0]) # pairwise left edges
    iy0 = np.maximum(b1[:, None, 1], b2[None, :, 1]) # pairwise top edges
    ix1 = np.minimum(b1[:, None, 2], b2[None, :, 2]) # pairwise right edges
    iy1 = np.minimum(b1[:, None, 3], b2[None, :, 3]) # pairwise bottom edges
    inter = np.clip(ix1-ix0, 0, None)*np.clip(iy1-iy0, 0, None) # pairwise intersection areas
    a1 = (b1[:, 2]-b1[:, 0])*(b1[:, 3]-b1[:, 1]) # areas of set 1
    a2 = (b2[:, 2]-b2[:, 0])*(b2[:, 3]-b2[:, 1]) # areas of set 2
    union = a1[:, None] + a2[None, :] - inter # pairwise unions
    return np.where(union > 0, inter/np.maximum(union, 1e-12), 0.0) # NxM IoU matrix
def nms(boxes, scores, thresh=0.5): # greedy NMS: keep best, suppress high-overlap rest
    order = [int(v) for v in np.argsort(-np.asarray(scores))] # indices sorted by score descending, as plain ints
    keep = [] # accepted box indices
    while order: # until every box is kept or suppressed
        i = order.pop(0) # take the current highest-scoring box
        keep.append(i) # keep it
        rest = [] # survivors of this round
        for j in order: # compare against all remaining boxes
            if iou_pair(boxes[i], boxes[j]) <= thresh: # low overlap -> survives
                rest.append(j) # keep for later rounds
        order = rest # continue with survivors only
    return keep # indices of kept boxes, best-first
nb = [[0.1,0.1,0.4,0.4],[0.12,0.12,0.42,0.42],[0.6,0.6,0.9,0.9]] # two overlapping + one far box
ns = [0.9, 0.75, 0.8] # middle box is a weaker duplicate of the first
nk = nms(nb, ns, 0.5) # run NMS
print('nms keep:', nk, '(expect [0, 2])') # duplicate index 1 must be suppressed
assert nk == [0, 2] # fail loudly if NMS logic is wrong
t0 = time.time() # start NMS timing
for _ in range(200): # repeat for a stable timing
    nms(nb, ns, 0.5) # tiny 3-box NMS
print(f'nms 3-box x200: {(time.time()-t0)*1000:.1f}ms') # pure-python overhead reference

In [ ]:
class TinyDet(nn.Module): # tiny backbone + box head + class head
    def __init__(self): # build layers
        super().__init__() # required init
        self.bb = nn.Sequential(nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2)) # 64->32->16->8 backbone
        self.fc = nn.Sequential(nn.Linear(64*8*8, 128), nn.ReLU()) # shared embedding
        self.box = nn.Linear(128, 4) # box head: predicts normalized xyxy
        self.cls = nn.Linear(128, NCLASS) # class head: one logit per shape
    def forward(self, x): # backbone then two heads
        h = self.bb(x) # (B,64,8,8) features
        h = self.fc(h.flatten(1)) # (B,128) embedding
        return torch.sigmoid(self.box(h)), self.cls(h) # sigmoid keeps boxes in [0,1]
det = TinyDet().to(DEVICE) # instantiate on device
print('params:', sum(p.numel() for p in det.parameters())) # log parameter count
opt = torch.optim.Adam(det.parameters(), lr=3e-3) # Adam for both heads
EPOCHS = 12 # small budget: synthetic task converges fast
hist = {'loss': [], 'miou': []} # history for CSV/plots
for ep in range(EPOCHS): # training loop
    det.train() # train mode
    ls, n = 0.0, 0 # loss accumulator + sample count
    t0 = time.time() # epoch timer
    for xb, bb, yb in tr_ld: # minibatches of (image, box, class)
        xb, bb, yb = xb.to(DEVICE), bb.to(DEVICE), yb.to(DEVICE) # move to device
        pb, pc = det(xb) # forward: predicted boxes + class logits
        loss = F.smooth_l1_loss(pb, bb) + F.cross_entropy(pc, yb) # box regression + classification
        opt.zero_grad() # clear old grads
        loss.backward() # backprop through both heads
        opt.step() # update weights
        ls += float(loss.detach())*xb.size(0) # accumulate (detach avoids the requires_grad warning)
        n += xb.size(0) # count
    det.eval() # eval mode for the val check
    ious = [] # per-sample IoUs on val
    correct, tot = 0, 0 # class accuracy counters
    with torch.no_grad(): # no grad for eval
        for xb, bb, yb in va_ld: # val batches
            pb, pc = det(xb.to(DEVICE)) # predict
            m = iou_matrix(pb.cpu().numpy(), bb.numpy()) # pairwise IoU (pred rows vs GT cols)
            ious += [m[i, i] for i in range(len(bb))] # diagonal = matched-pair IoUs
            correct += int(((pc.argmax(1).cpu()) == yb).sum()) # class hits
            tot += len(yb) # count
    miou = float(np.mean(ious)) # mean IoU over val
    acc = correct/tot # val class accuracy
    hist['loss'].append(ls/n) # record train loss
    hist['miou'].append(miou) # record val mIoU
    print(f'ep{ep+1}/{EPOCHS} loss={ls/n:.4f} val_mIoU={miou:.4f} val_acc={acc:.4f} {time.time()-t0:.1f}s') # progress

In [ ]:
det.eval() # freeze model for final eval
all_pb, all_bb, all_yc, all_yp = [], [], [], [] # buffers for full-val predictions
with torch.no_grad(): # no gradients needed
    for xb, bb, yb in va_ld: # sweep val set
        pb, pc = det(xb.to(DEVICE)) # predict boxes + classes
        all_pb.append(pb.cpu().numpy()) # store pred boxes
        all_bb.append(bb.numpy()) # store GT boxes
        all_yc.append(yb.numpy()) # store GT classes
        all_yp.append(pc.argmax(1).cpu().numpy()) # store pred classes
PB = np.concatenate(all_pb) # (NVAL,4) pred boxes
BB = np.concatenate(all_bb) # (NVAL,4) GT boxes
YC = np.concatenate(all_yc) # GT labels
YP = np.concatenate(all_yp) # pred labels
m = iou_matrix(PB, BB) # pairwise IoU matrix
diag = np.array([m[i, i] for i in range(len(BB))]) # matched IoUs
miou = float(diag.mean()) # headline mean IoU
p50 = float((diag >= 0.5).mean()) # precision@IoU0.5: fraction of good boxes
acc = float((YP == YC).mean()) # class accuracy
print(f'FINAL val mIoU={miou:.4f} prec@0.5={p50:.4f} cls_acc={acc:.4f}') # headline numbers
with open('./results/metrics.csv', 'w', newline='') as f: # save per-epoch history
    w = csv.writer(f) # need writer
    w.writerow(['epoch', 'train_loss', 'val_miou']) # header
    for i in range(EPOCHS): # one row per epoch
        w.writerow([i+1, f"{hist['loss'][i]:.4f}", f"{hist['miou'][i]:.4f}"]) # epoch stats
    w.writerow(['final', '', f'{miou:.4f}']) # headline row
fig, ax = plt.subplots(1, 2, figsize=(10, 4)) # loss + miou panels
ax[0].plot(hist['loss'], marker='o') # train loss curve
ax[0].set_title('train loss (box+cls)') # panel title
ax[0].set_xlabel('epoch') # x label
ax[1].plot(hist['miou'], marker='o') # val mIoU curve
ax[1].set_title('val mean IoU') # panel title
ax[1].set_xlabel('epoch') # x label
fig.suptitle(f'P8 TinyDet mIoU={miou:.3f} p@0.5={p50:.3f}') # headline title
fig.savefig('./results/curves.png', dpi=100) # save curves
plt.close(fig) # free memory
fig2, ax2 = plt.subplots(2, 4, figsize=(12, 6)) # 8 example overlays
Xs = (Xva.transpose(0, 2, 3, 1)) # CHW -> HWC for display
for k in range(8): # draw 8 val examples
    a = ax2.flat[k] # subplot handle
    a.imshow(np.clip(Xs[k], 0, 1)) # show the image
    g = BB[k]*IMG # GT box in pixels
    p = np.clip(PB[k], 0, 1)*IMG # pred box in pixels, clipped
    a.add_patch(patches.Rectangle((g[0], g[1]), g[2]-g[0], g[3]-g[1], fill=False, edgecolor='lime', linewidth=2)) # green = GT
    a.add_patch(patches.Rectangle((p[0], p[1]), p[2]-p[0], p[3]-p[1], fill=False, edgecolor='red', linewidth=1.5)) # red = pred
    a.set_title(f'gt={YC[k]} pr={YP[k]} iou={diag[k]:.2f}') # label with IoU
    a.axis('off') # hide axes
fig2.suptitle('green=GT red=pred') # legend as title
fig2.savefig('./results/boxes.png', dpi=100) # save overlay grid
plt.close(fig2) # free memory
print('done: see ./results/metrics.csv, curves.png, boxes.png') # artifact pointer

## What to write in your README (7 questions)
1. What did I build? 2. How do IoU / NMS / box regression work?
3. What mIoU / prec@0.5? 4. What surprised you? 5. Bottleneck?
6. What if NMS thresh / noise / 2 objects? 7. What next (anchors, multi-object, mAP)?